In [1]:
import os
import pickle
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
from datasets import load_from_disk

In [2]:
plt.rcParams['figure.figsize'] = (12, 10)
plt.rcParams['figure.dpi'] = 300

In [11]:
tokenized_data_str = 'hst_corpus_70m_32_None_None_None_None_None_shifted_log_knn_10'
tokenized_data_folder_path = f'/nfs/team361/sb75/DATASETS/gold/cell-graph-tokenizer/hst_corpus_70m/{tokenized_data_str}.dataset'
silver_folder_path = '/nfs/team361/sb75/DATASETS/silver/'
tokenizer_folder_path = '/nfs/team361/sb75/DATASETS/tokenizer'

In [4]:
# Load dataset
dataset = load_from_disk(tokenized_data_folder_path)

Loading dataset from disk:   0%|          | 0/32 [00:00<?, ?it/s]

In [6]:
# Define sample config
n_samples = 30
sample_type = 'frac_per_bin' # 'cells_per_bin', 'frac_per_bin', 'mixed'
avg_cells_per_bin = 50
sampled_cells_per_bin = 1
sample_frac = 0.1

In [7]:
# Retrieve cell IDs
dataset_cell_ids = dataset['cell_id']

In [8]:
# Create cell ID dict
cell_id_dict = {v: i for i, v in enumerate(dataset_cell_ids)} # for speedup

In [10]:
all_epochs_indices = [[] for i in range(n_samples)]

for subdir, _, files in os.walk(silver_folder_path + 'hst_corpus_70m'):
    for file_idx, file in enumerate(files):
        if file.endswith('.h5ad'):
            file_path = os.path.join(subdir, file)
            adata = sc.read_h5ad(file_path)
            print('====================================================================================================')
            print(f'Processing adata with file path {file_path}...')

            # Retrieve x and y bins
            print('Creating bins...')
            n_bins = int(np.round(len(adata) / avg_cells_per_bin, decimals=0))
            x_bins = np.linspace(adata.obsm['spatial'][:, 0].min(), adata.obsm['spatial'][:, 0].max(), int(np.round(np.sqrt(n_bins), decimals=0)) + 1)
            y_bins = np.linspace(adata.obsm['spatial'][:, 1].min(), adata.obsm['spatial'][:, 1].max(), int(np.round(np.sqrt(n_bins), decimals=0)) + 1)

            # Assign each point to a bin
            adata.obs['x_bin'] = pd.cut(adata.obsm['spatial'][:, 0], bins=x_bins, labels=False, include_lowest=True)
            adata.obs['y_bin'] = pd.cut(adata.obsm['spatial'][:, 1], bins=y_bins, labels=False, include_lowest=True)

            print('Sampling cells...')
            indices = [] 
            for i in range(n_samples):
                if sample_type == 'cells_per_bin':
                    sampled_df = adata.obs.groupby(['x_bin', 'y_bin']).apply(lambda group: group.sample(n=sampled_cells_per_bin))
                elif sample_type == 'frac_per_bin':
                    sampled_df = adata.obs.groupby(['x_bin', 'y_bin']).apply(lambda group: group.sample(frac=sample_frac))
                elif sample_type == 'mixed':
                    # First, sample 1 point from each group
                    min_sampled_df = adata.obs.groupby(['x_bin', 'y_bin']).apply(lambda group: group.sample(n=sampled_cells_per_bin))
                    
                    # Then, sample the remaining points based on sample_frac
                    remaining_df = adata.obs.groupby(['x_bin', 'y_bin']).apply(lambda group: group.sample(frac=sample_frac))
                    
                    # Combine both sampled DataFrames
                    sampled_df = pd.concat([min_sampled_df, remaining_df]).drop_duplicates()
                
                sampled_df = sampled_df.reset_index(drop=True)
            
                #adata.obs['sampled_flag'] = adata.obs['cell_id'].apply(lambda x: x in sampled_df['cell_id'].values.tolist())
    
                indices.append([cell_id_dict.get(item, None) for item in sampled_df['cell_id'].values.tolist()])
                del(sampled_df)
            
                #sc.pl.spatial(adata,
                #              color='sampled_flag',
                #              spot_size=spot_size)

            del(adata)
            for i, epoch_indices in enumerate(indices):
                all_epochs_indices[i].extend(epoch_indices)

            print('Saving results to disk...')
            with open(f'{tokenizer_folder_path}/{tokenized_data_str}_precomputed_epoch_split_{n_samples}.pkl', 'wb') as f:
                pickle.dump(all_epochs_indices, f)

all_epochs_n_nonzero_tokens = []

for epoch_indices in all_epochs_indices:
    epoch_data = dataset.select(epoch_indices)
    epoch_n_nonzero_tokens = epoch_data['n_nonzero_tokens']
    all_epochs_n_nonzero_tokens.append(epoch_n_nonzero_tokens)

with open(f'{tokenizer_folder_path}/{tokenized_data_str}_precomputed_epoch_n_nonzero_tokens_{n_samples}.pkl', 'wb') as f:
    pickle.dump(all_epochs_n_nonzero_tokens, f)

Processing adata with file path /nfs/team361/sb75/DATASETS/silver/hst_corpus_70m/xhb1002-AT16-15b_1p/AT16_3X__filt_cells_filt_genes_raw_log1pnorm_counts.h5ad...
Creating bins...
Sampling cells...
Saving results to disk...
Processing adata with file path /nfs/team361/sb75/DATASETS/silver/hst_corpus_70m/xhb1002-AT16-15b_1p/AT16-2D1__filt_cells_filt_genes_raw_log1pnorm_counts.h5ad...
Creating bins...
Sampling cells...
Saving results to disk...
Processing adata with file path /nfs/team361/sb75/DATASETS/silver/hst_corpus_70m/xhb1002-AT16-15b_1p/AT16-1C3__filt_cells_filt_genes_raw_log1pnorm_counts.h5ad...
Creating bins...
Sampling cells...
Saving results to disk...
Processing adata with file path /nfs/team361/sb75/DATASETS/silver/hst_corpus_70m/xhb1002-AT16-15b_1p/AT16-1B1__filt_cells_filt_genes_raw_log1pnorm_counts.h5ad...
Creating bins...
Sampling cells...
Saving results to disk...
Processing adata with file path /nfs/team361/sb75/DATASETS/silver/hst_corpus_70m/xhb1002-AT16-15b_1p/AT16-BRA

/software/cellgen/team361/sb75/envs/nichejepa/lib/python3.10/site-packages/anndata/_core/anndata.py:1754: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


Processing adata with file path /nfs/team361/sb75/DATASETS/silver/hst_corpus_70m/xhl1005-1b_1p/NSCLC_PT.h5ad...
Creating bins...
Sampling cells...
Saving results to disk...
Processing adata with file path /nfs/team361/sb75/DATASETS/silver/hst_corpus_70m/xhk1007-CV4-41b_1p/CV4-KID-0-FT-1-s3.h5ad...
Creating bins...
Sampling cells...
Saving results to disk...
Processing adata with file path /nfs/team361/sb75/DATASETS/silver/hst_corpus_70m/xhk1007-CV4-41b_1p/CV4-KID-0-FT-3-S3.h5ad...
Creating bins...
Sampling cells...
Saving results to disk...
Processing adata with file path /nfs/team361/sb75/DATASETS/silver/hst_corpus_70m/xhk1007-CV4-41b_1p/CV4-KID-0-FT-4-S3.h5ad...
Creating bins...
Sampling cells...
Saving results to disk...
Processing adata with file path /nfs/team361/sb75/DATASETS/silver/hst_corpus_70m/xhk1007-CV4-41b_1p/CV4-KID-0-FT-2-s3.h5ad...
Creating bins...
Sampling cells...
Saving results to disk...
Processing adata with file path /nfs/team361/sb75/DATASETS/silver/hst_corpus_70

/software/cellgen/team361/sb75/envs/nichejepa/lib/python3.10/site-packages/anndata/_core/anndata.py:1754: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


Processing adata with file path /nfs/team361/sb75/DATASETS/silver/hst_corpus_70m/xhln1004-1b_1p/NSCLC_healthy_lymph_node.h5ad...
Creating bins...
Sampling cells...
Saving results to disk...
Processing adata with file path /nfs/team361/sb75/DATASETS/silver/hst_corpus_70m/xhk1007-CV5-41b_1p/CV5-KID-0-FT-4-S3.h5ad...
Creating bins...
Sampling cells...
Saving results to disk...
Processing adata with file path /nfs/team361/sb75/DATASETS/silver/hst_corpus_70m/xhk1007-CV5-41b_1p/CV5-KID-0-FT-2-s3.h5ad...
Creating bins...
Sampling cells...
Saving results to disk...
Processing adata with file path /nfs/team361/sb75/DATASETS/silver/hst_corpus_70m/xhk1007-CV5-41b_1p/CV5-KID-0-FO-1.h5ad...
Creating bins...
Sampling cells...
Saving results to disk...
Processing adata with file path /nfs/team361/sb75/DATASETS/silver/hst_corpus_70m/xhk1007-CV5-41b_1p/CV5-KID-0-FT-1-s3.h5ad...
Creating bins...
Sampling cells...
Saving results to disk...
Processing adata with file path /nfs/team361/sb75/DATASETS/silver

/software/cellgen/team361/sb75/envs/nichejepa/lib/python3.10/site-packages/anndata/_core/anndata.py:1754: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


... (17 outputs left out)

/software/cellgen/team361/sb75/envs/nichejepa/lib/python3.10/site-packages/anndata/_core/anndata.py:1754: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


Processing adata with file path /nfs/team361/sb75/DATASETS/silver/hst_corpus_70m/xhs1000-39b_1p/adata_batch30.h5ad...
Creating bins...
Sampling cells...
Saving results to disk...
Processing adata with file path /nfs/team361/sb75/DATASETS/silver/hst_corpus_70m/xhs1000-39b_1p/adata_batch34.h5ad...
Creating bins...
Sampling cells...
Saving results to disk...
Processing adata with file path /nfs/team361/sb75/DATASETS/silver/hst_corpus_70m/xhs1000-39b_1p/adata_batch12.h5ad...
Creating bins...
Sampling cells...
Saving results to disk...
Processing adata with file path /nfs/team361/sb75/DATASETS/silver/hst_corpus_70m/xhs1000-39b_1p/adata_batch24.h5ad...
Creating bins...
Sampling cells...
Saving results to disk...
Processing adata with file path /nfs/team361/sb75/DATASETS/silver/hst_corpus_70m/xhk1007-CV7-41b_1p/CV7-KID-0-FT-4-S3.h5ad...
Creating bins...
Sampling cells...
Saving results to disk...
Processing adata with file path /nfs/team361/sb75/DATASETS/silver/hst_corpus_70m/xhk1007-CV7-41b_

/software/cellgen/team361/sb75/envs/nichejepa/lib/python3.10/site-packages/anndata/_core/anndata.py:1754: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


Processing adata with file path /nfs/team361/sb75/DATASETS/silver/hst_corpus_70m/xhb1003-1b_1p/breast_Xenium_no_exp_qv.h5ad...
Creating bins...
Sampling cells...
Saving results to disk...
Processing adata with file path /nfs/team361/sb75/DATASETS/silver/hst_corpus_70m/xhl41-2b_1p/Xenium_Preview_Human_Non_diseased_Lung_With_Add_on_FFPE_outs.h5ad...
Creating bins...
Sampling cells...
Saving results to disk...
Processing adata with file path /nfs/team361/sb75/DATASETS/silver/hst_corpus_70m/xhl41-2b_1p/Xenium_Preview_Human_Lung_Cancer_With_Add_on_2_FFPE_outs.h5ad...
Creating bins...
Sampling cells...
Saving results to disk...
Processing adata with file path /nfs/team361/sb75/DATASETS/silver/hst_corpus_70m/xhl29-1b_1p/Xenium_V1_hLung_cancer_section_outs.h5ad...
Creating bins...
Sampling cells...
Saving results to disk...
Processing adata with file path /nfs/team361/sb75/DATASETS/silver/hst_corpus_70m/xhs49-1b_1p/Xenium_Prime_Human_Skin_FFPE_outs.h5ad...
Creating bins...
Sampling cells...
Sa

In [12]:
# Sample batches
for file_path_sample_ratio in [0.01, 0.1, 0.25, 0.5]:
    n_samples = 30 / file_path_sample_ratio
    file_paths = []
    
    for subdir, _, files in os.walk(silver_folder_path + 'hst_corpus_70m'):
        for file_idx, file in enumerate(files):
            if file.endswith('.h5ad'):
                file_path = os.path.join(subdir, file)
                file_paths.append(file_path)
    
    file_path_sample_n = max(1, int(len(file_paths) * file_path_sample_ratio))
    sampled_file_paths = random.sample(file_paths, file_path_sample_n)
    
    all_epochs_indices = [[] for i in range(n_samples)]
    
    for subdir, _, files in os.walk(silver_folder_path + 'hst_corpus_70m'):
        for file_idx, file in enumerate(files):
            if file.endswith('.h5ad'):
                file_path = os.path.join(subdir, file)
                if file_path in sampled_file_paths:
                    adata = sc.read_h5ad(file_path)
                    print('====================================================================================================')
                    print(f'Processing adata with file path {file_path}...')
        
                    # Retrieve x and y bins
                    print('Creating bins...')
                    n_bins = int(np.round(len(adata) / avg_cells_per_bin, decimals=0))
                    x_bins = np.linspace(adata.obsm['spatial'][:, 0].min(), adata.obsm['spatial'][:, 0].max(), int(np.round(np.sqrt(n_bins), decimals=0)) + 1)
                    y_bins = np.linspace(adata.obsm['spatial'][:, 1].min(), adata.obsm['spatial'][:, 1].max(), int(np.round(np.sqrt(n_bins), decimals=0)) + 1)
        
                    # Assign each point to a bin
                    adata.obs['x_bin'] = pd.cut(adata.obsm['spatial'][:, 0], bins=x_bins, labels=False, include_lowest=True)
                    adata.obs['y_bin'] = pd.cut(adata.obsm['spatial'][:, 1], bins=y_bins, labels=False, include_lowest=True)
        
                    print('Sampling cells...')
                    indices = [] 
                    for i in range(n_samples):
                        if sample_type == 'cells_per_bin':
                            sampled_df = adata.obs.groupby(['x_bin', 'y_bin']).apply(lambda group: group.sample(n=sampled_cells_per_bin))
                        elif sample_type == 'frac_per_bin':
                            sampled_df = adata.obs.groupby(['x_bin', 'y_bin']).apply(lambda group: group.sample(frac=sample_frac))
                        elif sample_type == 'mixed':
                            # First, sample 1 point from each group
                            min_sampled_df = adata.obs.groupby(['x_bin', 'y_bin']).apply(lambda group: group.sample(n=sampled_cells_per_bin))
                            
                            # Then, sample the remaining points based on sample_frac
                            remaining_df = adata.obs.groupby(['x_bin', 'y_bin']).apply(lambda group: group.sample(frac=sample_frac))
                            
                            # Combine both sampled DataFrames
                            sampled_df = pd.concat([min_sampled_df, remaining_df]).drop_duplicates()
                        
                        sampled_df = sampled_df.reset_index(drop=True)
                    
                        #adata.obs['sampled_flag'] = adata.obs['cell_id'].apply(lambda x: x in sampled_df['cell_id'].values.tolist())
            
                        indices.append([cell_id_dict.get(item, None) for item in sampled_df['cell_id'].values.tolist()])
                        del(sampled_df)
                    
                        #sc.pl.spatial(adata,
                        #              color='sampled_flag',
                        #              spot_size=spot_size)
        
                    del(adata)
                    for i, epoch_indices in enumerate(indices):
                        all_epochs_indices[i].extend(epoch_indices)
        
                    print('Saving results to disk...')
                    with open(f'{tokenizer_folder_path}/{tokenized_data_str}_precomputed_epoch_split_{int(file_path_sample_ratio * 100)}_pct_subsample_{n_samples}.pkl', 'wb') as f:
                        pickle.dump(all_epochs_indices, f)
        
    all_epochs_n_nonzero_tokens = []
    
    for epoch_indices in all_epochs_indices:
        epoch_data = dataset.select(epoch_indices)
        epoch_n_nonzero_tokens = epoch_data['n_nonzero_tokens']
        all_epochs_n_nonzero_tokens.append(epoch_n_nonzero_tokens)
    
    with open(f'{tokenizer_folder_path}/{tokenized_data_str}_precomputed_epoch_n_nonzero_tokens_{int(file_path_sample_ratio * 100)}_pct_subsample_{n_samples}.pkl', 'wb') as f:
        pickle.dump(all_epochs_n_nonzero_tokens, f)

TypeError: 'float' object cannot be interpreted as an integer

In [ ]:
2+3

In [19]:
# Add missing epochs (don't change samples)
# for file_path_sample_ratio in [0.01, 0.1, 0.25, 0.5]:
for file_path_sample_ratio in [0.25]:

    with open(f'/lustre/scratch126/cellgen/team361/DATASETS/tokenizer/hst_corpus_70m_32_None_None_None_None_None_shifted_log_knn_10_precomputed_epoch_split_{int(file_path_sample_ratio * 100)}_pct_subsample_30.pkl', 'rb') as file:
        precomputed_epoch_splits = pickle.load(file)
    reverse_cell_id_dict = {value: key for key, value in cell_id_dict.items()}
    sampled_batches = set([reverse_cell_id_dict[item].split('_')[0] + "_" + reverse_cell_id_dict[item].split('_')[1] for item in precomputed_epoch_splits[0]])
    
    all_epochs_indices = [[] for i in range(n_samples)]
    
    for subdir, _, files in os.walk(silver_folder_path + 'hst_corpus_70m'):
        for file_idx, file in enumerate(files):
            if file.endswith('.h5ad'):
                file_path = os.path.join(subdir, file)
                adata = sc.read_h5ad(file_path)
                if adata.obs['cell_id'].apply(lambda x: x.split('_')[0] + '_' + x.split('_')[1]).values.tolist()[0] in sampled_batches:
                    print('====================================================================================================')
                    print(f'Processing adata with file path {file_path}...')
        
                    # Retrieve x and y bins
                    print('Creating bins...')
                    n_bins = int(np.round(len(adata) / avg_cells_per_bin, decimals=0))
                    x_bins = np.linspace(adata.obsm['spatial'][:, 0].min(), adata.obsm['spatial'][:, 0].max(), int(np.round(np.sqrt(n_bins), decimals=0)) + 1)
                    y_bins = np.linspace(adata.obsm['spatial'][:, 1].min(), adata.obsm['spatial'][:, 1].max(), int(np.round(np.sqrt(n_bins), decimals=0)) + 1)
        
                    # Assign each point to a bin
                    adata.obs['x_bin'] = pd.cut(adata.obsm['spatial'][:, 0], bins=x_bins, labels=False, include_lowest=True)
                    adata.obs['y_bin'] = pd.cut(adata.obsm['spatial'][:, 1], bins=y_bins, labels=False, include_lowest=True)
        
                    print('Sampling cells...')
                    indices = [] 
                    for i in range(n_samples):
                        if sample_type == 'cells_per_bin':
                            sampled_df = adata.obs.groupby(['x_bin', 'y_bin']).apply(lambda group: group.sample(n=sampled_cells_per_bin))
                        elif sample_type == 'frac_per_bin':
                            sampled_df = adata.obs.groupby(['x_bin', 'y_bin']).apply(lambda group: group.sample(frac=sample_frac))
                        elif sample_type == 'mixed':
                            # First, sample 1 point from each group
                            min_sampled_df = adata.obs.groupby(['x_bin', 'y_bin']).apply(lambda group: group.sample(n=sampled_cells_per_bin))
                            
                            # Then, sample the remaining points based on sample_frac
                            remaining_df = adata.obs.groupby(['x_bin', 'y_bin']).apply(lambda group: group.sample(frac=sample_frac))
                            
                            # Combine both sampled DataFrames
                            sampled_df = pd.concat([min_sampled_df, remaining_df]).drop_duplicates()
                        
                        sampled_df = sampled_df.reset_index(drop=True)
                    
                        #adata.obs['sampled_flag'] = adata.obs['cell_id'].apply(lambda x: x in sampled_df['cell_id'].values.tolist())
            
                        indices.append([cell_id_dict.get(item, None) for item in sampled_df['cell_id'].values.tolist()])
                        del(sampled_df)
                    
                        #sc.pl.spatial(adata,
                        #              color='sampled_flag',
                        #              spot_size=spot_size)
        
                    del(adata)
                    for i, epoch_indices in enumerate(indices):
                        all_epochs_indices[i].extend(epoch_indices)
        
                    print('Saving results to disk...')
                    with open(f'/lustre/scratch126/cellgen/team361/DATASETS/tokenizer/{tokenized_data_str}_precomputed_epoch_split_{int(file_path_sample_ratio * 100)}_pct_subsample_{n_samples}.pkl', 'wb') as f:
                        pickle.dump(all_epochs_indices, f)
        
    all_epochs_n_nonzero_tokens = []
    
    for epoch_indices in all_epochs_indices:
        epoch_data = dataset.select(epoch_indices)
        epoch_n_nonzero_tokens = epoch_data['n_nonzero_tokens']
        all_epochs_n_nonzero_tokens.append(epoch_n_nonzero_tokens)
    
    with open(f'/lustre/scratch126/cellgen/team361/DATASETS/tokenizer/{tokenized_data_str}_precomputed_epoch_n_nonzero_tokens_{int(file_path_sample_ratio * 100)}_pct_subsample_{n_samples}.pkl', 'wb') as f:
        pickle.dump(all_epochs_n_nonzero_tokens, f)

Processing adata with file path /lustre/scratch126/cellgen/team361/DATASETS/silver/hst_corpus_70m/xhb1002-AT23-9b_1p/AT23-BRA-3-FO-1E2-1__filt_cells_filt_genes_raw_log1pnorm_counts.h5ad...
Creating bins...
Sampling cells...
Saving results to disk...
Processing adata with file path /lustre/scratch126/cellgen/team361/DATASETS/silver/hst_corpus_70m/xhb1002-AT23-9b_1p/AT23-BRA-3-FO-1B1-1-S26__filt_cells_filt_genes_raw_log1pnorm_counts.h5ad...
Creating bins...
Sampling cells...
Saving results to disk...
Processing adata with file path /lustre/scratch126/cellgen/team361/DATASETS/silver/hst_corpus_70m/mhl80-8b_1p/adata_batch3.h5ad...
Creating bins...
Sampling cells...
Saving results to disk...
Processing adata with file path /lustre/scratch126/cellgen/team361/DATASETS/silver/hst_corpus_70m/xho47-S2-1b_1p/Xenium_Prime_Human_Ovary_FF_outs_part2.h5ad...
Creating bins...
Sampling cells...
Saving results to disk...
Processing adata with file path /lustre/scratch126/cellgen/team361/DATASETS/silver/

/lustre/scratch126/cellgen/team361/sb75/conda/envs/nichejepa_new/lib/python3.10/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


Processing adata with file path /lustre/scratch126/cellgen/team361/DATASETS/silver/hst_corpus_70m/xhk1007-CV5-41b_1p/CV5-KID-0-FT-2-s3.h5ad...
Creating bins...
Sampling cells...
Saving results to disk...


/lustre/scratch126/cellgen/team361/sb75/conda/envs/nichejepa_new/lib/python3.10/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/lustre/scratch126/cellgen/team361/sb75/conda/envs/nichejepa_new/lib/python3.10/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/lustre/scratch126/cellgen/team361/sb75/conda/envs/nichejepa_new/lib/python3.10/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


Processing adata with file path /lustre/scratch126/cellgen/team361/DATASETS/silver/hst_corpus_70m/xhk1007-CV5-41b_1p/CV5-KID-0-FT-1-s3.h5ad...
Creating bins...
Sampling cells...
Saving results to disk...
Processing adata with file path /lustre/scratch126/cellgen/team361/DATASETS/silver/hst_corpus_70m/xhd52-1b_1p/Xenium_V1_Human_Ductal_Adenocarcinoma_FFPE_outs.h5ad...
Creating bins...
Sampling cells...
Saving results to disk...
Processing adata with file path /lustre/scratch126/cellgen/team361/DATASETS/silver/hst_corpus_70m/xhsc44-6b_1p/adata_batch1.h5ad...
Creating bins...
Sampling cells...
Saving results to disk...
Processing adata with file path /lustre/scratch126/cellgen/team361/DATASETS/silver/hst_corpus_70m/xhsc44-6b_1p/adata_batch5.h5ad...
Creating bins...
Sampling cells...
Saving results to disk...
Processing adata with file path /lustre/scratch126/cellgen/team361/DATASETS/silver/hst_corpus_70m/xhc1001-1b_1p/NicheCompass_hs_XeniumAdultColon_AM_07112024_132853_raw.h5ad...
Creatin

/lustre/scratch126/cellgen/team361/sb75/conda/envs/nichejepa_new/lib/python3.10/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/lustre/scratch126/cellgen/team361/sb75/conda/envs/nichejepa_new/lib/python3.10/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


... (49 outputs left out)

/lustre/scratch126/cellgen/team361/sb75/conda/envs/nichejepa_new/lib/python3.10/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/lustre/scratch126/cellgen/team361/sb75/conda/envs/nichejepa_new/lib/python3.10/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/lustre/scratch126/cellgen/team361/sb75/conda/envs/nichejepa_new/lib/python3.10/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


Processing adata with file path /lustre/scratch126/cellgen/team361/DATASETS/silver/hst_corpus_70m/xhk1007-CV1-41b_1p/CV1-KID-0-FT-2.h5ad...
Creating bins...
Sampling cells...
Saving results to disk...


/lustre/scratch126/cellgen/team361/sb75/conda/envs/nichejepa_new/lib/python3.10/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


Processing adata with file path /lustre/scratch126/cellgen/team361/DATASETS/silver/hst_corpus_70m/xhk1007-CV1-41b_1p/CV1-KID-0-FO-1.h5ad...
Creating bins...
Sampling cells...
Saving results to disk...
Processing adata with file path /lustre/scratch126/cellgen/team361/DATASETS/silver/hst_corpus_70m/xhe1008-4-20b_1p/section051_SOB26.h5ad...
Creating bins...
Sampling cells...
Saving results to disk...
Processing adata with file path /lustre/scratch126/cellgen/team361/DATASETS/silver/hst_corpus_70m/xhb1002-AT13-2b_1p/AT13-BRA-3-FO-4__filt_cells_filt_genes_raw_log1pnorm_counts.h5ad...
Creating bins...
Sampling cells...
Saving results to disk...
Processing adata with file path /lustre/scratch126/cellgen/team361/DATASETS/silver/hst_corpus_70m/xhb37-3b_1p/Xenium_V1_hBoneMarrow_acute_lymphoid_leukemia_section_outs.h5ad...
Creating bins...
Sampling cells...
Saving results to disk...
Processing adata with file path /lustre/scratch126/cellgen/team361/DATASETS/silver/hst_corpus_70m/xhl40-2b_2p/Xeni

In [20]:
file_path_sample_ratio = 0.25
n_samples = int(1 / file_path_sample_ratio) * 10

with open(f'/lustre/scratch126/cellgen/team361/DATASETS/tokenizer/hst_corpus_70m_32_None_None_None_None_None_shifted_log_knn_10_precomputed_epoch_split_{int(file_path_sample_ratio * 100)}_pct_subsample_{n_samples}.pkl', 'rb') as file:
    precomputed_epoch_split_subsample_full = pickle.load(file)

with open(f'/lustre/scratch126/cellgen/team361/DATASETS/tokenizer/hst_corpus_70m_32_None_None_None_None_None_shifted_log_knn_10_precomputed_epoch_n_nonzero_tokens_{int(file_path_sample_ratio * 100)}_pct_subsample_{n_samples}.pkl', 'rb') as file:
    precomputed_epoch_n_nonzero_tokens_subsample_full = pickle.load(file)

with open(f'/lustre/scratch126/cellgen/team361/DATASETS/tokenizer/hst_corpus_70m_32_None_None_None_None_None_shifted_log_knn_10_precomputed_epoch_split_{int(file_path_sample_ratio * 100)}_pct_subsample_30.pkl', 'rb') as file:
    precomputed_epoch_split_subsample_30 = pickle.load(file)

with open(f'/lustre/scratch126/cellgen/team361/DATASETS/tokenizer/hst_corpus_70m_32_None_None_None_None_None_shifted_log_knn_10_precomputed_epoch_n_nonzero_tokens_{int(file_path_sample_ratio * 100)}_pct_subsample_30.pkl', 'rb') as file:
    precomputed_epoch_n_nonzero_tokens_subsample_30 = pickle.load(file)

In [21]:
len(precomputed_epoch_split_subsample_full)

40

In [22]:
len(precomputed_epoch_n_nonzero_tokens_subsample_full)

40

In [23]:
len(precomputed_epoch_split_subsample_full[0])

1435118

In [24]:
len(precomputed_epoch_n_nonzero_tokens_subsample_full[0])

1435118

In [25]:
len(precomputed_epoch_split_subsample_full[-1])

1435118

In [26]:
len(precomputed_epoch_n_nonzero_tokens_subsample_full[-1])

1435118

In [27]:
len(precomputed_epoch_split_subsample_30[0])

1385045

In [28]:
len(precomputed_epoch_n_nonzero_tokens_subsample_30[0])

1385045

In [29]:
precomputed_epoch_split_subsample_full_new = precomputed_epoch_split_subsample_30 + precomputed_epoch_split_subsample_full[30:]
precomputed_epoch_n_nonzero_tokens_subsample_full_new = precomputed_epoch_n_nonzero_tokens_subsample_30 + precomputed_epoch_n_nonzero_tokens_subsample_full[30:]

In [20]:
with open(f'/lustre/scratch126/cellgen/team361/DATASETS/tokenizer/{tokenized_data_str}_precomputed_epoch_split_{int(file_path_sample_ratio * 100)}_pct_subsample_{n_samples}.pkl', 'wb') as f:
    pickle.dump(precomputed_epoch_split_subsample_full_new, f)

with open(f'/lustre/scratch126/cellgen/team361/DATASETS/tokenizer/{tokenized_data_str}_precomputed_epoch_n_nonzero_tokens_{int(file_path_sample_ratio * 100)}_pct_subsample_{n_samples}.pkl', 'wb') as f:
    pickle.dump(precomputed_epoch_n_nonzero_tokens_subsample_full_new, f)

In [21]:
with open(f'/lustre/scratch126/cellgen/team361/DATASETS/tokenizer/{tokenized_data_str}_precomputed_epoch_split_{int(file_path_sample_ratio * 100)}_pct_subsample_{n_samples}.pkl', 'rb') as file:
    precomputed_epoch_split_subsample_full_new = pickle.load(file)

with open(f'/lustre/scratch126/cellgen/team361/DATASETS/tokenizer/{tokenized_data_str}_precomputed_epoch_n_nonzero_tokens_{int(file_path_sample_ratio * 100)}_pct_subsample_{n_samples}.pkl', 'rb') as file:
    precomputed_epoch_n_nonzero_tokens_subsample_full_new = pickle.load(file)

In [22]:
len(precomputed_epoch_split_subsample_full_new)

100

In [24]:
len(precomputed_epoch_n_nonzero_tokens_subsample_full_new)

100

In [25]:
len(precomputed_epoch_split_subsample_full_new[-1])

585028

In [26]:
len(precomputed_epoch_n_nonzero_tokens_subsample_full_new[-1])

585028